# BfmS–CHEMBL7029 Molecular Dynamics Simulation (100 ns)

**Fully automated** — click `Runtime > Run All` to execute the entire pipeline.

This notebook:
1. Installs all dependencies (OpenMM, RDKit, MDTraj, etc.)
2. Clones the repo from GitHub
3. Prepares the protein-ligand complex
4. Runs 100 ns NPT molecular dynamics on GPU
5. Analyzes trajectory (RMSD, contacts, H-bonds)
6. Generates publication figures
7. Saves results to Google Drive
8. Pushes results back to GitHub

**Expected runtime:** ~12-24 hours on T4 GPU (~50K atom system)

## 0. Setup & Dependencies

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/bfms-md-results'
!mkdir -p {DRIVE_DIR}

In [ ]:
# Install dependencies
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# After condacolab restart, install packages via conda
!conda install -y -c conda-forge openmm cudatoolkit rdkit mdtraj pdbfixer meeko numpy pandas matplotlib 2>&1 | tail -5
!pip install -q admet-ai 2>&1 | tail -3

# Verify GPU
import openmm as mm
for i in range(mm.Platform.getNumPlatforms()):
    p = mm.Platform.getPlatform(i)
    print(f'Platform: {p.getName()}')
    if p.getName() == 'CUDA':
        print(f'  Devices: {p.getPropertyDefaultValue("DeviceIndex")}')

In [ ]:
# Clone repository
import os
os.chdir('/content')
if not os.path.exists('bfms-drug-discovery'):
    !git clone https://github.com/arian-gogani/bfms-drug-discovery.git
os.chdir('bfms-drug-discovery')
!git pull
print(f'Working directory: {os.getcwd()}')

## 1. Prepare Protein-Ligand Complex

In [ ]:
import numpy as np
import openmm as mm
import openmm.app as app
import openmm.unit as unit
from pdbfixer import PDBFixer
from rdkit import Chem
from rdkit.Chem import AllChem
from pathlib import Path
import time, sys

# Configuration
SIM_TIME_NS = 100
TIMESTEP_FS = 2.0
TEMPERATURE_K = 300.0
PRESSURE_ATM = 1.0
PADDING_NM = 1.2
IONIC_STRENGTH_M = 0.15
SAVE_INTERVAL_PS = 100    # trajectory frame every 100 ps (1000 frames total)
LOG_INTERVAL_PS = 10      # energy log every 10 ps
CHECKPOINT_INTERVAL_PS = 5000  # checkpoint every 5 ns

MD_DIR = Path('results/md')
MD_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = Path('results/figures')

SMILES = 'C/C(=C\\Cn1oc(=O)[nH]c1=O)c1cccc(OCc2nc(-c3ccc(C(F)(F)F)cc3)oc2C)c1'

# MMFF94 unit conversions (Halgren 1996)
BOND_K_CONV = 143.9325 * 4.184 * 100        # md/A -> kJ/mol/nm2
ANGLE_K_CONV = 0.043844 * (180/np.pi)**2 * 4.184  # md*A/rad2 (deg) -> kJ/mol/rad2
TORSION_K_CONV = 4.184 / 2.0                 # kcal/mol -> kJ/mol with /2

LJ_PARAMS = {
    'H': (0.1069, 0.0657), 'C': (0.1908, 0.3598), 'N': (0.1824, 0.7113),
    'O': (0.1661, 0.8786), 'F': (0.1750, 0.2552), 'S': (0.2000, 1.0460),
}

print('Configuration loaded.')

In [ ]:
# 1a. Create ligand with docked coordinates
print('Creating ligand from SMILES with docked coordinates...')
mol = Chem.MolFromSmiles(SMILES)
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
AllChem.MMFFOptimizeMolecule(mol)

# Parse PDBQT docked pose
pdbqt_coords = []
smiles_map = {}
with open('results/poses/CHEMBL7029_pose.pdbqt') as f:
    model = 0
    for line in f:
        if line.startswith('MODEL'): model = int(line.split()[1])
        if model != 1: continue
        if line.startswith('ENDMDL'): break
        if line.startswith('ATOM'):
            pdbqt_coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
        if 'SMILES IDX' in line:
            parts = line.split('SMILES IDX')[1].split()
            for i in range(0, len(parts), 2):
                smiles_map[int(parts[i])-1] = int(parts[i+1])-1
pdbqt_coords = np.array(pdbqt_coords)

# Map heavy atom coordinates from docked pose
conf = mol.GetConformer()
heavy_idx = 0
mapped = set()
for ai in range(mol.GetNumAtoms()):
    if mol.GetAtomWithIdx(ai).GetAtomicNum() == 1: continue
    if heavy_idx in smiles_map:
        pi = smiles_map[heavy_idx]
        if pi < len(pdbqt_coords):
            conf.SetAtomPosition(ai, pdbqt_coords[pi].tolist())
            mapped.add(ai)
    heavy_idx += 1

# Optimize H positions only
mmff_props = AllChem.MMFFGetMoleculeProperties(mol)
ff_opt = AllChem.MMFFGetMoleculeForceField(mol, mmff_props)
for idx in mapped: ff_opt.AddFixedPoint(idx)
ff_opt.Minimize(maxIts=200)

print(f'Ligand: {mol.GetNumAtoms()} atoms, {len(mapped)} heavy atoms positioned from docked pose')
lig_coords_nm = np.array([[conf.GetAtomPosition(i).x/10, conf.GetAtomPosition(i).y/10, conf.GetAtomPosition(i).z/10] for i in range(mol.GetNumAtoms())])

In [ ]:
# 1b. Prepare protein (Chain C from 3KLN)
print('Preparing protein Chain C...')
chain_c_path = MD_DIR / 'chain_c.pdb'
lines = [l for l in open('data/3KLN_clean.pdb') if l.startswith('ATOM') and l[21] == 'C']
with open(chain_c_path, 'w') as f:
    for l in lines: f.write(l)
    f.write(f'TER   {int(lines[-1][6:11])+1:5d}      {lines[-1][17:20]} C{lines[-1][22:26]}\n')
    f.write('END\n')

fixer = PDBFixer(filename=str(chain_c_path))
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixed_path = MD_DIR / 'chain_c_fixed.pdb'
with open(fixed_path, 'w') as f:
    app.PDBFile.writeFile(fixer.topology, fixer.positions, f)

forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
protein_pdb = app.PDBFile(str(fixed_path))
modeller = app.Modeller(protein_pdb.topology, protein_pdb.positions)
modeller.addHydrogens(forcefield, pH=7.0)
print(f'Protein with H: {modeller.topology.getNumAtoms()} atoms')

# Solvate
modeller.addSolvent(forcefield, model='tip3p', padding=PADDING_NM*unit.nanometers,
                     ionicStrength=IONIC_STRENGTH_M*unit.molar, positiveIon='Na+', negativeIon='Cl-')
print(f'Solvated: {modeller.topology.getNumAtoms()} atoms')

# Remove clashing waters
pos_nm = np.array(modeller.positions.value_in_unit(unit.nanometers))
clash_res = [r for r in modeller.topology.residues() if r.name == 'HOH'
             and any(np.min(np.linalg.norm(lig_coords_nm - pos_nm[a.index], axis=1)) < 0.25
                     for a in r.atoms())]
if clash_res:
    modeller.delete(clash_res)
    print(f'Removed {len(clash_res)} clashing waters -> {modeller.topology.getNumAtoms()} atoms')

In [ ]:
# 1c. Create system and inject ligand forces
print('Creating OpenMM system...')
system = forcefield.createSystem(modeller.topology, nonbondedMethod=app.PME,
                                  nonbondedCutoff=1.0*unit.nanometers,
                                  constraints=app.HBonds, hydrogenMass=None)

n_prot_water = system.getNumParticles()

# Add ligand particles
lig_indices = []
for i in range(mol.GetNumAtoms()):
    system.addParticle(mol.GetAtomWithIdx(i).GetMass() * unit.amu)
    lig_indices.append(n_prot_water + i)
    # Add dummy NB params (will be overridden)
    for force in system.getForces():
        if isinstance(force, mm.NonbondedForce):
            force.addParticle(0, 0.1, 0)
            break

# Find forces
nb = bond_f = angle_f = torsion_f = None
for force in system.getForces():
    if isinstance(force, mm.NonbondedForce): nb = force
    elif isinstance(force, mm.HarmonicBondForce): bond_f = force
    elif isinstance(force, mm.HarmonicAngleForce): angle_f = force
    elif isinstance(force, mm.PeriodicTorsionForce): torsion_f = force

# Set ligand nonbonded params
for ri, ti in enumerate(lig_indices):
    chg = mmff_props.GetMMFFPartialCharge(ri)
    sig, eps = LJ_PARAMS.get(mol.GetAtomWithIdx(ri).GetSymbol(), (0.1908, 0.3598))
    nb.setParticleParameters(ti, chg, sig, eps)

# Bonds + 1-2 exclusions
excluded = set()
for bond in mol.GetBonds():
    ri, rj = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    ti, tj = lig_indices[ri], lig_indices[rj]
    p = mmff_props.GetMMFFBondStretchParams(mol, ri, rj)
    if p:
        a1, a2 = mol.GetAtomWithIdx(ri), mol.GetAtomWithIdx(rj)
        if a1.GetAtomicNum() == 1 or a2.GetAtomicNum() == 1:
            system.addConstraint(ti, tj, p[2]/10.0)  # Constrain H-X bonds for 2fs
        else:
            bond_f.addBond(ti, tj, p[2]/10.0, p[1]*BOND_K_CONV)
    pair = (min(ti,tj), max(ti,tj))
    if pair not in excluded:
        nb.addException(ti, tj, 0, 1, 0); excluded.add(pair)

# Angles + 1-3 exclusions
for i in range(mol.GetNumAtoms()):
    nbrs = [n.GetIdx() for n in mol.GetAtomWithIdx(i).GetNeighbors()]
    for ni in range(len(nbrs)):
        for nj in range(ni+1, len(nbrs)):
            a1, a2, a3 = nbrs[ni], i, nbrs[nj]
            p = mmff_props.GetMMFFAngleBendParams(mol, a1, a2, a3)
            if p:
                angle_f.addAngle(lig_indices[a1], lig_indices[a2], lig_indices[a3],
                                 p[2]*np.pi/180, p[1]*ANGLE_K_CONV)
            pair = (min(lig_indices[nbrs[ni]], lig_indices[nbrs[nj]]),
                    max(lig_indices[nbrs[ni]], lig_indices[nbrs[nj]]))
            if pair not in excluded:
                nb.addException(pair[0], pair[1], 0, 1, 0); excluded.add(pair)

# Torsions + 1-4 scaled interactions
for bond in mol.GetBonds():
    i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    for ni in mol.GetAtomWithIdx(i).GetNeighbors():
        if ni.GetIdx() == j: continue
        for nj in mol.GetAtomWithIdx(j).GetNeighbors():
            if nj.GetIdx() == i: continue
            a1, a4 = ni.GetIdx(), nj.GetIdx()
            p = mmff_props.GetMMFFTorsionParams(mol, a1, i, j, a4)
            if p:
                for per, v, bp in [(1,p[1],0.0), (2,p[2],np.pi), (3,p[3],0.0)]:
                    if abs(v) > 1e-6:
                        phase = bp if v >= 0 else (np.pi - bp)
                        torsion_f.addTorsion(lig_indices[a1], lig_indices[i],
                                             lig_indices[j], lig_indices[a4],
                                             per, phase, abs(v)*TORSION_K_CONV)
            t1, t4 = lig_indices[a1], lig_indices[a4]
            pair = (min(t1,t4), max(t1,t4))
            if pair not in excluded:
                q1 = mmff_props.GetMMFFPartialCharge(a1)
                q4 = mmff_props.GetMMFFPartialCharge(a4)
                s1, e1 = LJ_PARAMS.get(mol.GetAtomWithIdx(a1).GetSymbol(), (0.1908,0.3598))
                s4, e4 = LJ_PARAMS.get(mol.GetAtomWithIdx(a4).GetSymbol(), (0.1908,0.3598))
                nb.addException(t1, t4, q1*q4*0.8333, (s1+s4)/2, (e1*e4)**0.5*0.5)
                excluded.add(pair)

# Add ligand to topology
chain = modeller.topology.addChain('B')
res = modeller.topology.addResidue('UNL', chain)
ec = {}
lig_topo_atoms = []
for i in range(mol.GetNumAtoms()):
    elem = mol.GetAtomWithIdx(i).GetSymbol()
    ec[elem] = ec.get(elem, 0) + 1
    a = modeller.topology.addAtom(f'{elem}{ec[elem]}', app.element.Element.getBySymbol(elem), res)
    lig_topo_atoms.append(a)
for bond in mol.GetBonds():
    modeller.topology.addBond(lig_topo_atoms[bond.GetBeginAtomIdx()], lig_topo_atoms[bond.GetEndAtomIdx()])

# Barostat
system.addForce(mm.MonteCarloBarostat(PRESSURE_ATM*unit.atmospheres, TEMPERATURE_K*unit.kelvin, 25))

# Combine positions
prot_pos = modeller.positions.value_in_unit(unit.nanometers)
all_pos = [mm.Vec3(*p) for p in prot_pos] + [mm.Vec3(*c) for c in lig_coords_nm]
positions = all_pos * unit.nanometers

topology = modeller.topology
print(f'Final system: {system.getNumParticles()} particles, {len(excluded)} ligand exclusions')

# Save topology
with open(MD_DIR / 'solvated_system.pdb', 'w') as f:
    app.PDBFile.writeFile(topology, positions, f)
with open(MD_DIR / 'system.xml', 'w') as f:
    f.write(mm.XmlSerializer.serialize(system))

## 2. Run MD Simulation (100 ns)

In [ ]:
# Select best platform (prefer CUDA on Colab)
platform = None
for pname in ['CUDA', 'OpenCL', 'CPU']:
    try:
        platform = mm.Platform.getPlatformByName(pname)
        props = {'Precision': 'mixed'} if pname in ('CUDA', 'OpenCL') else {}
        test_sys = mm.System(); test_sys.addParticle(1.0)
        test_int = mm.VerletIntegrator(0.001)
        ctx = mm.Context(test_sys, test_int, platform, props)
        del ctx, test_int, test_sys
        print(f'Using platform: {pname}')
        break
    except: platform = None
if platform is None:
    platform = mm.Platform.getPlatformByName('CPU')
    props = {}
    print('Fallback: CPU')

# Check for checkpoint to resume
RESUME = (MD_DIR / 'checkpoint.chk').exists()

integrator = mm.LangevinMiddleIntegrator(
    TEMPERATURE_K*unit.kelvin, 1/unit.picoseconds, TIMESTEP_FS*unit.femtoseconds)
simulation = app.Simulation(topology, system, integrator, platform, props)

if RESUME:
    print('Resuming from checkpoint...')
    simulation.loadCheckpoint(str(MD_DIR / 'checkpoint.chk'))
else:
    simulation.context.setPositions(positions)

    # Minimize
    print('Energy minimization...')
    state = simulation.context.getState(getEnergy=True)
    print(f'  Before: {state.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole):.0f} kJ/mol')
    simulation.minimizeEnergy(maxIterations=10000)
    state = simulation.context.getState(getEnergy=True)
    print(f'  After:  {state.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole):.0f} kJ/mol')

    # NVT equilibration (100 ps)
    print('NVT equilibration (100 ps)...')
    simulation.context.setVelocitiesToTemperature(TEMPERATURE_K*unit.kelvin)
    simulation.step(int(100e3 / TIMESTEP_FS))

    # NPT equilibration (500 ps)
    print('NPT equilibration (500 ps)...')
    simulation.step(int(500e3 / TIMESTEP_FS))
    state = simulation.context.getState(getEnergy=True)
    print(f'  Equilibrated energy: {state.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole):.0f} kJ/mol')

# Save production topology
state = simulation.context.getState(getPositions=True)
with open(MD_DIR / 'production_topology.pdb', 'w') as f:
    app.PDBFile.writeFile(topology, state.getPositions(), f)

print('Setup complete. Starting production...')

In [ ]:
# Production MD (100 ns)
total_steps = int(SIM_TIME_NS * 1e6 / TIMESTEP_FS)
save_steps = int(SAVE_INTERVAL_PS * 1e3 / TIMESTEP_FS)
log_steps = int(LOG_INTERVAL_PS * 1e3 / TIMESTEP_FS)
chk_steps = int(CHECKPOINT_INTERVAL_PS * 1e3 / TIMESTEP_FS)

simulation.reporters.append(app.DCDReporter(str(MD_DIR / 'production.dcd'), save_steps))
simulation.reporters.append(app.StateDataReporter(
    str(MD_DIR / 'production_log.csv'), log_steps,
    time=True, potentialEnergy=True, kineticEnergy=True,
    totalEnergy=True, temperature=True, volume=True,
    density=True, speed=True, separator=','))
simulation.reporters.append(app.CheckpointReporter(str(MD_DIR / 'checkpoint.chk'), chk_steps))
simulation.reporters.append(app.StateDataReporter(
    sys.stdout, chk_steps, time=True, potentialEnergy=True,
    temperature=True, speed=True, remainingTime=True, totalSteps=total_steps))

# Also checkpoint to Drive periodically
simulation.reporters.append(app.CheckpointReporter(f'{DRIVE_DIR}/checkpoint.chk', chk_steps))

print(f'Production: {SIM_TIME_NS} ns ({total_steps:,} steps, {TIMESTEP_FS} fs)')
t0 = time.time()
simulation.step(total_steps)
elapsed = time.time() - t0
print(f'\nDone! {elapsed/3600:.1f} hours, {SIM_TIME_NS/(elapsed/86400):.1f} ns/day')

# Save final state
simulation.saveCheckpoint(str(MD_DIR / 'final_checkpoint.chk'))
state = simulation.context.getState(getPositions=True)
with open(MD_DIR / 'final_state.pdb', 'w') as f:
    app.PDBFile.writeFile(topology, state.getPositions(), f)

## 3. Trajectory Analysis

In [ ]:
import mdtraj as md
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import csv

plt.rcParams.update({'font.size': 10, 'figure.dpi': 300})

POCKET_RESIDUES = [66, 69, 72, 98, 101, 153, 154, 155, 212, 215, 216]

# Load trajectory
print('Loading trajectory...')
traj = md.load(str(MD_DIR / 'production.dcd'), top=str(MD_DIR / 'production_topology.pdb'))
print(f'  {traj.n_frames} frames, {traj.n_atoms} atoms, {traj.time[0]:.0f}-{traj.time[-1]:.0f} ps')

# Identify selections
protein_atoms = traj.topology.select('protein')
backbone = traj.topology.select('backbone')
ligand_atoms = traj.topology.select('resname UNL')
if len(ligand_atoms) == 0:
    ligand_atoms = traj.topology.select('not protein and not water and not (resname Na+ Cl- NA CL HOH WAT)')
ligand_heavy = np.array([i for i in ligand_atoms if traj.topology.atom(i).element.symbol != 'H'])
pocket_atoms = np.array([a.index for r in traj.topology.residues
                          if r.is_protein and r.resSeq in POCKET_RESIDUES for a in r.atoms])

print(f'  Protein: {len(protein_atoms)}, Ligand: {len(ligand_atoms)} ({len(ligand_heavy)} heavy), Pocket: {len(pocket_atoms)}')

In [ ]:
# Compute RMSD
print('Computing RMSD...')
time_ns = traj.time / 1000.0
protein_rmsd = md.rmsd(traj, traj, frame=0, atom_indices=backbone) * 10  # A
traj_aligned = traj.superpose(traj, frame=0, atom_indices=backbone)
ligand_rmsd = md.rmsd(traj_aligned, traj_aligned, frame=0, atom_indices=ligand_heavy) * 10
pocket_rmsd = md.rmsd(traj_aligned, traj_aligned, frame=0, atom_indices=pocket_atoms) * 10 if len(pocket_atoms) > 0 else None

eq = len(protein_rmsd) // 4  # last 75% as equilibrated
print(f'  Protein RMSD: {np.mean(protein_rmsd[eq:]):.2f} +/- {np.std(protein_rmsd[eq:]):.2f} A')
print(f'  Ligand RMSD:  {np.mean(ligand_rmsd[eq:]):.2f} +/- {np.std(ligand_rmsd[eq:]):.2f} A')

In [ ]:
# Compute contacts and H-bonds
print('Computing contacts...')
contact_cutoff = 0.45  # nm
residue_contacts = {}
prot_res = {r.resSeq: [a.index for a in r.atoms if a.element.symbol != 'H']
            for r in traj.topology.residues if r.is_protein}

for resSeq, res_atoms in prot_res.items():
    pairs = np.array([[ra, la] for ra in res_atoms for la in ligand_heavy])
    if len(pairs) == 0: continue
    dists = md.compute_distances(traj, pairs)
    min_d = dists.min(axis=1)
    frac = np.mean(min_d < contact_cutoff)
    if frac > 0.1:
        residue_contacts[resSeq] = {'fraction': frac, 'min_distances': min_d}

total_contacts = np.zeros(traj.n_frames)
for data in residue_contacts.values():
    total_contacts += (data['min_distances'] < contact_cutoff).astype(float)

print(f'  Residues with >10% contact: {len(residue_contacts)}')
print(f'  Mean contacts/frame: {np.mean(total_contacts):.1f}')

# H-bonds (sample every 10th frame for speed)
print('Computing H-bonds...')
lig_set, prot_set = set(ligand_atoms.tolist()), set(protein_atoms.tolist())
hbond_counts = []
hbond_residues = {}
for fi in range(0, traj.n_frames, max(1, traj.n_frames//200)):
    hbs = md.baker_hubbard(traj[fi], freq=0.0)
    count = 0
    for d, h, a in hbs:
        if (d in lig_set and a in prot_set) or (d in prot_set and a in lig_set):
            count += 1
            pa = a if d in lig_set else d
            r = traj.topology.atom(pa).residue
            key = f'{r.name}{r.resSeq}'
            hbond_residues[key] = hbond_residues.get(key, 0) + 1
    hbond_counts.append(count)

n_hb_samples = len(hbond_counts)
for k in hbond_residues: hbond_residues[k] /= n_hb_samples
hbond_counts = np.array(hbond_counts)
print(f'  Mean H-bonds/frame: {np.mean(hbond_counts):.2f}')

# Ligand RMSF
rmsf = md.rmsf(traj_aligned, traj_aligned, frame=0, atom_indices=ligand_heavy) * 10

# Pocket Rg
if len(pocket_atoms) > 0:
    rog = md.compute_rg(traj.atom_slice(pocket_atoms)) * 10
else:
    rog = None

In [ ]:
# Load energy log
energy_data = {'time_ns':[], 'potential':[], 'kinetic':[], 'total':[], 'temperature':[], 'density':[]}
with open(MD_DIR / 'production_log.csv') as f:
    for row in csv.DictReader(f):
        try:
            t = float(list(row.values())[0])
            energy_data['time_ns'].append(t/1000)
            energy_data['potential'].append(float(list(row.values())[1]))
            energy_data['kinetic'].append(float(list(row.values())[2]))
            energy_data['total'].append(float(list(row.values())[3]))
            energy_data['temperature'].append(float(list(row.values())[4]))
            energy_data['density'].append(float(list(row.values())[6]))
        except: pass
for k in energy_data: energy_data[k] = np.array(energy_data[k])
print(f'Energy log: {len(energy_data["time_ns"])} entries')

## 4. Generate Publication Figures

In [ ]:
COL = {'prot':'#2C3E50', 'lig':'#E74C3C', 'pocket':'#3498DB', 'hb':'#27AE60', 'cont':'#F39C12'}
w = max(1, len(time_ns)//100)

# Figure 5: Binding Stability Dashboard
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('CHEMBL7029-BfmS MD: Binding Stability (100 ns)', fontsize=13, fontweight='bold', y=0.98)

ax = axes[0,0]
ax.plot(time_ns, protein_rmsd, color=COL['prot'], alpha=0.3, lw=0.5)
ax.plot(time_ns, ligand_rmsd, color=COL['lig'], alpha=0.3, lw=0.5)
ax.plot(time_ns, np.convolve(protein_rmsd, np.ones(w)/w, 'same'), color=COL['prot'], lw=2, label='Protein backbone')
ax.plot(time_ns, np.convolve(ligand_rmsd, np.ones(w)/w, 'same'), color=COL['lig'], lw=2, label='Ligand')
if pocket_rmsd is not None:
    ax.plot(time_ns, np.convolve(pocket_rmsd, np.ones(w)/w, 'same'), color=COL['pocket'], lw=2, label='Pocket')
ax.set_xlabel('Time (ns)'); ax.set_ylabel('RMSD (A)'); ax.set_title('A. RMSD', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlim(0, time_ns[-1])

ax = axes[0,1]
ax.plot(time_ns, np.convolve(total_contacts, np.ones(w)/w, 'same'), color=COL['cont'], lw=1.5, label='Contacts')
ax2 = ax.twinx()
hb_time = np.linspace(time_ns[0], time_ns[-1], len(hbond_counts))
ax2.plot(hb_time, hbond_counts, 'o', color=COL['hb'], ms=2, alpha=0.5, label='H-bonds')
ax.set_xlabel('Time (ns)'); ax.set_ylabel('Contacts', color=COL['cont'])
ax2.set_ylabel('H-bonds', color=COL['hb']); ax.set_title('B. Interactions', fontweight='bold')

ax = axes[1,0]
if rog is not None:
    ax.plot(time_ns, rog, color=COL['pocket'], alpha=0.3, lw=0.5)
    ax.plot(time_ns, np.convolve(rog, np.ones(w)/w, 'same'), color=COL['pocket'], lw=2)
    ax.axhline(np.mean(rog), color=COL['pocket'], ls='--', alpha=0.5)
    ax.text(0.95, 0.95, f'Mean: {np.mean(rog):.1f} +/- {np.std(rog):.1f} A', transform=ax.transAxes, ha='right', va='top', fontsize=9)
ax.set_xlabel('Time (ns)'); ax.set_ylabel('Rg (A)'); ax.set_title('C. Pocket Compactness', fontweight='bold')

ax = axes[1,1]
ax.hist(protein_rmsd[eq:], 50, alpha=0.5, color=COL['prot'], density=True,
        label=f'Protein ({np.mean(protein_rmsd[eq:]):.2f}+/-{np.std(protein_rmsd[eq:]):.2f} A)')
ax.hist(ligand_rmsd[eq:], 50, alpha=0.5, color=COL['lig'], density=True,
        label=f'Ligand ({np.mean(ligand_rmsd[eq:]):.2f}+/-{np.std(ligand_rmsd[eq:]):.2f} A)')
ax.set_xlabel('RMSD (A)'); ax.set_ylabel('Density'); ax.set_title('D. RMSD Distribution', fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig(FIG_DIR / 'fig5_md_binding_stability.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(f'{DRIVE_DIR}/fig5_md_binding_stability.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Figure 5 saved.')

In [ ]:
# Figure 6: Residue Interactions
from matplotlib.gridspec import GridSpec
fig = plt.figure(figsize=(14, 6))
gs = GridSpec(1, 3, width_ratios=[2, 1, 1], wspace=0.35)

sorted_contacts = sorted(residue_contacts.items(), key=lambda x: x[1]['fraction'], reverse=True)[:20]
labels, fracs, hb_fracs = [], [], []
for resSeq, data in sorted_contacts:
    for r in traj.topology.residues:
        if r.resSeq == resSeq and r.is_protein:
            lbl = f'{r.name}{resSeq}'
            labels.append(lbl); fracs.append(data['fraction'])
            hb_fracs.append(hbond_residues.get(lbl, 0)); break

ax = fig.add_subplot(gs[0])
y = np.arange(len(labels))
ax.barh(y, fracs, color=COL['cont'], alpha=0.7, label='VdW contact')
ax.barh(y, hb_fracs, color=COL['hb'], alpha=0.8, label='H-bond')
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8); ax.invert_yaxis()
ax.set_xlabel('Frequency'); ax.set_title('A. Per-Residue Interactions', fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
for i, lbl in enumerate(labels):
    rn = int(''.join(c for c in lbl if c.isdigit()))
    if rn in POCKET_RESIDUES:
        ax.get_yticklabels()[i].set_fontweight('bold')
        ax.get_yticklabels()[i].set_color(COL['pocket'])

ax = fig.add_subplot(gs[1])
colors_rmsf = [COL['lig'] if r < np.mean(rmsf) else '#C0392B' for r in rmsf]
ax.bar(range(len(rmsf)), rmsf, color=colors_rmsf, alpha=0.8)
ax.axhline(np.mean(rmsf), color='gray', ls='--', alpha=0.5, label=f'Mean: {np.mean(rmsf):.2f} A')
ax.set_xlabel('Heavy Atom Index'); ax.set_ylabel('RMSF (A)'); ax.set_title('B. Ligand Flexibility', fontweight='bold')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[2])
if traj.n_frames > 4:
    qs = ['Q1\n0-25ns','Q2\n25-50ns','Q3\n50-75ns','Q4\n75-100ns']
    qsz = traj.n_frames // 4
    for idx, (resSeq, data) in enumerate(sorted_contacts[:5]):
        qf = [np.mean(data['min_distances'][q*qsz:(q+1)*qsz] < 0.45) for q in range(4)]
        for r in traj.topology.residues:
            if r.resSeq == resSeq and r.is_protein:
                ax.plot(range(4), qf, 'o-', lw=2, ms=6, label=f'{r.name}{resSeq}', alpha=0.8); break
    ax.set_xticks(range(4)); ax.set_xticklabels(qs, fontsize=8)
    ax.set_ylabel('Contact Frequency'); ax.set_ylim(0, 1.05)
    ax.set_title('C. Contact Stability', fontweight='bold'); ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig6_residue_interactions.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(f'{DRIVE_DIR}/fig6_residue_interactions.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Figure 6 saved.')

In [ ]:
# Figure 7: Simulation Diagnostics
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('MD Simulation Diagnostics', fontsize=13, fontweight='bold', y=0.98)
t = energy_data['time_ns']; ew = max(1, len(t)//50)

ax = axes[0,0]
ax.plot(t, energy_data['total']/1000, color=COL['prot'], lw=0.5, alpha=0.3)
ax.plot(t, np.convolve(energy_data['total']/1000, np.ones(ew)/ew, 'same'), color=COL['prot'], lw=2)
ax.set_xlabel('Time (ns)'); ax.set_ylabel('Total Energy (x10^3 kJ/mol)'); ax.set_title('A. Energy', fontweight='bold')

ax = axes[0,1]
ax.plot(t, energy_data['temperature'], color=COL['lig'], lw=0.5, alpha=0.2)
ax.plot(t, np.convolve(energy_data['temperature'], np.ones(ew)/ew, 'same'), color=COL['lig'], lw=2)
ax.axhline(300, color='gray', ls='--', alpha=0.5); ax.set_ylim(280,320)
ax.set_xlabel('Time (ns)'); ax.set_ylabel('Temperature (K)'); ax.set_title('B. Temperature', fontweight='bold')

ax = axes[1,0]
ax.plot(t, energy_data['density'], color=COL['pocket'], lw=0.5, alpha=0.2)
ax.plot(t, np.convolve(energy_data['density'], np.ones(ew)/ew, 'same'), color=COL['pocket'], lw=2)
ax.set_xlabel('Time (ns)'); ax.set_ylabel('Density (g/mL)'); ax.set_title('C. Density', fontweight='bold')

ax = axes[1,1]
ax.plot(t, energy_data['potential']/1000, color=COL['prot'], lw=0.5, alpha=0.2, label='Potential')
ax.plot(t, energy_data['kinetic']/1000, color=COL['lig'], lw=0.5, alpha=0.2, label='Kinetic')
ax.plot(t, np.convolve(energy_data['potential']/1000, np.ones(ew)/ew, 'same'), color=COL['prot'], lw=2)
ax.plot(t, np.convolve(energy_data['kinetic']/1000, np.ones(ew)/ew, 'same'), color=COL['lig'], lw=2)
ax.set_xlabel('Time (ns)'); ax.set_ylabel('Energy (x10^3 kJ/mol)'); ax.set_title('D. Energy Components', fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig(FIG_DIR / 'fig7_md_diagnostics.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(f'{DRIVE_DIR}/fig7_md_diagnostics.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Figure 7 saved.')

In [ ]:
# Save summary statistics
with open(MD_DIR / 'md_summary_statistics.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Metric', 'Full_Mean', 'Full_SD', 'Eq_Mean', 'Eq_SD'])
    w.writerow(['Protein_Backbone_RMSD_A', f'{np.mean(protein_rmsd):.3f}', f'{np.std(protein_rmsd):.3f}',
                f'{np.mean(protein_rmsd[eq:]):.3f}', f'{np.std(protein_rmsd[eq:]):.3f}'])
    w.writerow(['Ligand_RMSD_A', f'{np.mean(ligand_rmsd):.3f}', f'{np.std(ligand_rmsd):.3f}',
                f'{np.mean(ligand_rmsd[eq:]):.3f}', f'{np.std(ligand_rmsd[eq:]):.3f}'])
    w.writerow(['H_Bonds_Per_Frame', f'{np.mean(hbond_counts):.3f}', f'{np.std(hbond_counts):.3f}', '', ''])
    w.writerow(['Contacts_Per_Frame', f'{np.mean(total_contacts):.3f}', f'{np.std(total_contacts):.3f}', '', ''])

with open(MD_DIR / 'residue_contact_frequencies.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Residue', 'Contact_Freq', 'HBond_Freq', 'Is_Pocket'])
    for resSeq, data in sorted(residue_contacts.items(), key=lambda x: x[1]['fraction'], reverse=True):
        name = next((f'{r.name}{resSeq}' for r in traj.topology.residues if r.resSeq == resSeq and r.is_protein), str(resSeq))
        w.writerow([name, f'{data["fraction"]:.4f}', f'{hbond_residues.get(name,0):.4f}', resSeq in POCKET_RESIDUES])

print('Summary statistics saved.')

## 5. Save to Google Drive & Push to GitHub

In [ ]:
# Copy key results to Drive
import shutil
for f in ['md_summary_statistics.csv', 'residue_contact_frequencies.csv', 'production_log.csv',
          'final_state.pdb', 'production_topology.pdb']:
    src = MD_DIR / f
    if src.exists():
        shutil.copy2(src, f'{DRIVE_DIR}/{f}')

# Copy trajectory (can be large)
dcd = MD_DIR / 'production.dcd'
if dcd.exists():
    shutil.copy2(dcd, f'{DRIVE_DIR}/production.dcd')

print(f'Results saved to Google Drive: {DRIVE_DIR}')
!ls -lh {DRIVE_DIR}/

In [ ]:
# Push to GitHub (requires auth token)
# Set your GitHub token as a Colab secret or paste it below
import os

# Try Colab secrets first
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except:
    GITHUB_TOKEN = ''  # Paste your token here if not using secrets

if GITHUB_TOKEN:
    !git config user.email 'colab@users.noreply.github.com'
    !git config user.name 'Colab MD Runner'
    !git remote set-url origin https://{GITHUB_TOKEN}@github.com/arian-gogani/bfms-drug-discovery.git
    !git add results/md/md_summary_statistics.csv results/md/residue_contact_frequencies.csv \
           results/md/production_log.csv results/md/final_state.pdb results/md/production_topology.pdb \
           results/figures/fig5_md_binding_stability.png results/figures/fig6_residue_interactions.png \
           results/figures/fig7_md_diagnostics.png
    !git commit -m 'Add 100ns MD simulation results (CHEMBL7029-BfmS)' \
        -m 'Co-Authored-By: Claude Opus 4.6 (1M context) <noreply@anthropic.com>'
    !git push origin main
    print('Pushed to GitHub!')
else:
    print('No GITHUB_TOKEN found. To push, add your token as a Colab secret named GITHUB_TOKEN.')
    print('Or run: git push origin main (after authenticating manually)')

In [ ]:
print('='*70)
print('SIMULATION COMPLETE')
print('='*70)
print(f'Trajectory: {traj.n_frames} frames, {SIM_TIME_NS} ns')
print(f'Protein RMSD (eq): {np.mean(protein_rmsd[eq:]):.2f} +/- {np.std(protein_rmsd[eq:]):.2f} A')
print(f'Ligand RMSD (eq):  {np.mean(ligand_rmsd[eq:]):.2f} +/- {np.std(ligand_rmsd[eq:]):.2f} A')
print(f'Mean H-bonds:      {np.mean(hbond_counts):.2f}')
print(f'Mean contacts:     {np.mean(total_contacts):.1f}')
print(f'\nResults saved to:')
print(f'  Google Drive: {DRIVE_DIR}')
print(f'  Repository:   results/md/ and results/figures/')
print('='*70)